<a href="https://colab.research.google.com/github/prof-atritiack/2TDSPO-CP2-GEN_AI/blob/main/CP2_ia_generativa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CHECKPOINT 02: Assistente de IA Generativa com Hugging Face e Gemini

**Disciplina:** Disruptive Architectures: IoT, IoB & Generative AI

**Grupo:**
- Nome 1 (RM)
- Nome 2 (RM)
- Nome 3 (RM)

**Tema escolhido:** (escreva aqui o tema da lista do README)

---

Nesta avaliação o grupo vai aprofundar o assistente construído na atividade anterior, em 4 etapas (e 1 bônus):

1. Assistente com guardrails (restrições no `system`)
2. Comparação com temperatura
3. Chat com memória e troca de modelo
4. Interface web com Gradio e controle de temperatura
5. API com sessões independentes

Os trechos marcados com **`# >>> PERSONALIZE`** devem ser alterados pelo grupo.  
Execute as células em ordem, de cima para baixo.

## 0. Configuração

In [ ]:
!pip install huggingface_hub google-genai gradio -q

In [ ]:
from huggingface_hub import InferenceClient
from google import genai
from google.genai import types
from google.colab import userdata

In [ ]:
HF_TOKEN = userdata.get("HF_TOKEN")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("HF_TOKEN ok" if HF_TOKEN else "ERRO: adicione HF_TOKEN nos Secrets do Colab")
print("GEMINI_API_KEY ok" if GEMINI_API_KEY else "ERRO: adicione GEMINI_API_KEY nos Secrets do Colab")

In [ ]:
MODELO_HF     = "meta-llama/Llama-3.1-8B-Instruct"
MODELO_GEMINI = "gemini-2.5-flash-lite"

cliente_hf     = InferenceClient(model=MODELO_HF, token=HF_TOKEN, provider="auto")
cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
# >>> PERSONALIZE: system prompt do tema do grupo.
# Um bom system diz: quem o assistente é, sobre o que fala,
# como responde (idioma, tamanho) e o que não deve fazer.
SYSTEM_PROMPT = """Você é um assistente de casa inteligente.
Ajude o usuário com dúvidas sobre automação residencial, sensores e dispositivos conectados.
Responda sempre em português, de forma clara, em no máximo 5 frases.
Se a pergunta não tiver relação com casa inteligente, diga educadamente que não pode ajudar."""

---
## Etapa 1: Assistente com guardrails

Na atividade anterior o grupo testou personalidades diferentes.
Aqui o foco é o **guardrail**: usar o `system` para restringir o assistente a responder **apenas** perguntas do tema.

A mesma pergunta será enviada com três versões do `system`:
- sem restrição
- com restrição de tema
- com restrição de tema e de formato

Também será testada uma pergunta **fora do escopo** para ver se o assistente respeita o guardrail.

In [ ]:
def perguntar_hf(pergunta, system, temperatura=0.7):
    mensagens = [
        {"role": "system", "content": system},
        {"role": "user",   "content": pergunta},
    ]
    resposta = cliente_hf.chat_completion(
        messages=mensagens,
        max_tokens=300,
        temperature=temperatura,
    )
    return resposta.choices[0].message.content

In [ ]:
# >>> PERSONALIZE: adapte os três systems e as perguntas ao tema do grupo.
versoes_system = {
    "Sem restrição": "Você é um assistente prestativo. Responda em português.",

    "Com restrição de tema": """Você é um assistente de casa inteligente.
Se a pergunta não for sobre automação residencial, diga que não pode ajudar.
Responda em português.""",

    "Com restrição de tema e formato": """Você é um assistente de casa inteligente.
Se a pergunta não for sobre automação residencial, diga que não pode ajudar.
Responda em português, em no máximo 3 frases, sem usar listas ou tópicos.""",
}

# >>> PERSONALIZE: uma pergunta dentro do escopo e uma fora.
pergunta_dentro = "Como um sensor de presença ajuda a economizar energia?"
pergunta_fora   = "Qual a capital da França?"

for nome, system in versoes_system.items():
    print(f"\n{'='*55}")
    print(f"  {nome}")
    print(f"{'='*55}")
    print(f"\n[Dentro do escopo] {pergunta_dentro}")
    print(perguntar_hf(pergunta_dentro, system))
    print(f"\n[Fora do escopo]   {pergunta_fora}")
    print(perguntar_hf(pergunta_fora, system))

**Observações do grupo (Etapa 1):**

- O assistente sem restrição respondeu a pergunta fora do escopo?
- Com o guardrail de tema, ele recusou? A recusa foi educada?
- A restrição de formato mudou o estilo da resposta?

(responda aqui)

---
## Etapa 2: Comparação com temperatura

Na atividade anterior o grupo comparou Hugging Face e Gemini com o mesmo prompt.
Aqui o grupo compara o efeito da **temperatura** nos dois modelos.

- `temperature=0.0` — o modelo escolhe sempre o token mais provável
- `temperature=1.0` — o modelo amostra com mais aleatoriedade

A mesma pergunta é enviada com cada valor de temperatura para os dois modelos.

In [ ]:
def perguntar_gemini(pergunta, system, temperatura=1.0):
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,
            max_output_tokens=300,
            temperature=temperatura,
        ),
    )
    return resposta.text

In [ ]:
# >>> PERSONALIZE: 2 perguntas sobre o tema do grupo.
perguntas = [
    "Quais sensores são mais usados em uma casa inteligente?",
    "Quais cuidados de segurança devo ter com câmeras conectadas?",
]

for p in perguntas:
    print(f"\nPERGUNTA: {p}")
    for temp in [0.0, 1.0]:
        print(f"\n--- temperature={temp} ---")
        print(f"[Hugging Face] {perguntar_hf(p, SYSTEM_PROMPT, temperatura=temp)}")
        print(f"[Gemini]       {perguntar_gemini(p, SYSTEM_PROMPT, temperatura=temp)}")
    print("\n" + "=" * 55)

**Observações do grupo (Etapa 2):**

- Com `temperature=0.0`, as respostas dos dois modelos ficaram parecidas entre si?
- Com `temperature=1.0`, as respostas ficaram mais variadas?
- Qual combinação de modelo e temperatura vocês consideram melhor para o tema?

(responda aqui)

---
## Etapa 3: Chat com memória e troca de modelo

Na atividade anterior o chat usava apenas o Hugging Face.
Aqui o grupo pode trocar o modelo durante a conversa, mantendo o histórico.

Comandos:
- `sair` — encerra o chat
- `limpar` — apaga o histórico
- `historico` — mostra quantas mensagens estão guardadas
- `modelo hf` — troca para Hugging Face
- `modelo gemini` — troca para Gemini

**Teste sugerido:** diga seu nome, pergunte "qual é o meu nome?", troque o modelo e pergunte de novo.

In [ ]:
def responder_chat(historico, modelo):
    if modelo == "hf":
        resposta = cliente_hf.chat_completion(messages=historico, max_tokens=300)
        return resposta.choices[0].message.content

    # Gemini: converte histórico (sem system) para o formato correto
    conteudos = []
    for msg in historico:
        if msg["role"] == "system":
            continue  # system vai em system_instruction, não aqui
        papel = "model" if msg["role"] == "assistant" else "user"
        conteudos.append({"role": papel, "parts": [{"text": msg["content"]}]})

    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=conteudos,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=300,
        ),
    )
    return resposta.text

In [ ]:
historico    = [{"role": "system", "content": SYSTEM_PROMPT}]
modelo_atual = "hf"

print(f"Chat iniciado com modelo: {modelo_atual}")
print("Comandos: sair | limpar | historico | modelo hf | modelo gemini\n")

while True:
    entrada = input("Você: ")

    if entrada.lower() == "sair":
        print("Encerrando chat.")
        break

    if entrada.lower() == "limpar":
        historico = [{"role": "system", "content": SYSTEM_PROMPT}]
        print("\n(histórico apagado)\n")
        continue

    if entrada.lower() == "historico":
        print(f"\n(mensagens: {len(historico)} | modelo: {modelo_atual})\n")
        continue

    if entrada.lower() in ("modelo hf", "modelo gemini"):
        modelo_atual = entrada.split()[1]
        print(f"\n(modelo trocado para: {modelo_atual})\n")
        continue

    historico.append({"role": "user", "content": entrada})

    texto = responder_chat(historico, modelo_atual)
    historico.append({"role": "assistant", "content": texto})

    print(f"\n[{modelo_atual}] {texto}\n")

In [ ]:
for msg in historico:
    print(f"[{msg['role']}] {msg['content'][:80].replace(chr(10), ' ')}")

**Observações do grupo (Etapa 3):**

- Ao trocar de modelo no meio da conversa, o novo modelo "lembrou" do que foi dito?
- Por que isso acontece (ou não acontece)?
- O que muda na forma como o histórico é enviado para o Gemini em relação ao Hugging Face?

(responda aqui)

---
## Etapa 4: Interface web com Gradio e controle de temperatura

Na atividade anterior a interface tinha o seletor de modelo.
Aqui o grupo adiciona um **slider de temperatura** para controlar a criatividade das respostas em tempo real.

In [ ]:
import gradio as gr

def texto_da_mensagem(conteudo):
    if isinstance(conteudo, str):
        return conteudo
    if isinstance(conteudo, list):
        return " ".join(item.get("text", "") for item in conteudo if isinstance(item, dict))
    return str(conteudo)


def responder(mensagem, historico_gradio, provedor, temperatura):
    if provedor == "Hugging Face":
        mensagens = [{"role": "system", "content": SYSTEM_PROMPT}]
        for msg in historico_gradio:
            mensagens.append({"role": msg["role"], "content": texto_da_mensagem(msg["content"])})
        mensagens.append({"role": "user", "content": mensagem})

        resposta = cliente_hf.chat_completion(
            messages=mensagens,
            max_tokens=300,
            temperature=temperatura,
        )
        return resposta.choices[0].message.content

    else:
        conteudos = []
        for msg in historico_gradio:
            papel = "model" if msg["role"] == "assistant" else "user"
            conteudos.append({"role": papel, "parts": [{"text": texto_da_mensagem(msg["content"])}]})
        conteudos.append({"role": "user", "parts": [{"text": mensagem}]})

        resposta = cliente_gemini.models.generate_content(
            model=MODELO_GEMINI,
            contents=conteudos,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                max_output_tokens=300,
                temperature=temperatura,
            ),
        )
        return resposta.text

In [ ]:
# >>> PERSONALIZE: título, descrição e exemplos de acordo com o tema do grupo.
gr.ChatInterface(
    fn=responder,
    additional_inputs=[
        gr.Radio(["Hugging Face", "Gemini"], value="Hugging Face", label="Modelo"),
        gr.Slider(minimum=0.0, maximum=1.0, step=0.1, value=0.7, label="Temperatura"),
    ],
    title="Assistente de Casa Inteligente",
    description="Pergunte sobre automação residencial, sensores e dispositivos conectados.",
    examples=[
        ["Como funciona uma lâmpada inteligente?", "Hugging Face", 0.7],
        ["Vale a pena usar tomadas conectadas?",   "Gemini",       0.3],
    ],
).launch(share=True)

**Observações do grupo (Etapa 4):**

- Tire um print da interface com o slider visível e coloque no README do repositório.
- Teste a mesma pergunta com temperatura 0.0 e 1.0 na interface. O que mudou?
- Qual combinação de modelo e temperatura funcionou melhor para o tema?

(responda aqui)

> Para parar a interface, interrompa a célula (botão de parar do Colab).

---
## Etapa 5: API com sessões independentes

Na atividade anterior a API não guardava histórico entre chamadas.
Aqui cada `session_id` tem seu próprio histórico — duas sessões paralelas não compartilham contexto.

In [ ]:
%%writefile app_avaliacao.py
import os
from fastapi import FastAPI
from pydantic import BaseModel
from huggingface_hub import InferenceClient

# Token e system prompt vêm de variáveis de ambiente (nunca escreva o token no código)
client = InferenceClient(
    model="meta-llama/Llama-3.1-8B-Instruct",
    token=os.environ["HF_TOKEN"],
    provider="auto",
)
SYSTEM_PROMPT = os.environ.get("SYSTEM_PROMPT", "Você é um assistente prestativo.")

app = FastAPI()

# Histórico por sessão
sessoes: dict[str, list] = {}

class Pergunta(BaseModel):
    session_id: str
    mensagem: str

@app.post("/chat")
def chat(pergunta: Pergunta):
    if pergunta.session_id not in sessoes:
        sessoes[pergunta.session_id] = [{"role": "system", "content": SYSTEM_PROMPT}]

    historico = sessoes[pergunta.session_id]
    historico.append({"role": "user", "content": pergunta.mensagem})

    resposta = client.chat_completion(messages=historico, max_tokens=300)
    texto = resposta.choices[0].message.content

    historico.append({"role": "assistant", "content": texto})

    return {"session_id": pergunta.session_id, "resposta": texto}

@app.delete("/sessao/{session_id}")
def limpar_sessao(session_id: str):
    sessoes.pop(session_id, None)
    return {"status": "sessão apagada", "session_id": session_id}

In [ ]:
!pip install fastapi uvicorn -q

import os, subprocess, time
os.environ["HF_TOKEN"]      = HF_TOKEN
os.environ["SYSTEM_PROMPT"] = SYSTEM_PROMPT

servidor = subprocess.Popen(["uvicorn", "app_avaliacao:app", "--port", "8000"])
time.sleep(5)
print("Servidor rodando em http://localhost:8000")

In [ ]:
import requests

# Sessão A — apresenta o nome
requests.post("http://localhost:8000/chat", json={"session_id": "sessao_a", "mensagem": "Meu nome é Ana."})
r_a = requests.post("http://localhost:8000/chat", json={"session_id": "sessao_a", "mensagem": "Qual é o meu nome?"})

# Sessão B — não apresentou o nome
r_b = requests.post("http://localhost:8000/chat", json={"session_id": "sessao_b", "mensagem": "Qual é o meu nome?"})

print("Sessão A:", r_a.json()["resposta"])
print("Sessão B:", r_b.json()["resposta"])

In [ ]:
servidor.terminate()
print("Servidor encerrado.")

**Observações do grupo (Etapa 5 — Bônus):**

- A sessão A respondeu o nome corretamente? E a sessão B?
- O que mudou nessa API em relação à da atividade anterior?
- Cite um caso de uso real onde sessões independentes seriam necessárias.

(responda aqui)